# einops-einsum — ex7: multi-head attention scores — split (b, s, h*d) and contract per head

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-einsum`. Running the final beacon cell reports progress against the `Einops: Deep Learning` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Deep Learning` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-einsum`** (exercise 7). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-einsum"
DD_SUBTOPIC = "Einops: Deep Learning"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.einsum — quick refresher

`einsum(*tensors, pattern)` performs sum-contraction over named indices:
1. **Elementwise** — `'i j, i j -> i j'` multiplies pointwise (no reduction).
2. **Matmul** — `'i k, k j -> i j'` contracts the shared `k` (sum-reduce).
3. **Batched** — `'b i k, b k j -> b i j'` carries `b` through, contracts `k`.
4. **Three operands** — `'i j, j k, k l -> i l'` chains two contractions; the optimizer picks pairing order.

**The two rules:**
- An index that appears on input AND output → preserved (broadcast-like).
- An index that appears on input but NOT on output → sum-contracted.

### Exercise 7 — multi-head attention scores — split (b, s, h*d) and contract per head

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Split a packed (B, S, H*D) projection into per-head tensors via rearrange, then use einsum to compute (B, H, S, S) attention scores in one pattern, printing intermediate shapes.
> Keywords: multi-head, split, rearrange, contraction, debug-print
> ```

**KCs targeted:** `einsum-batched`, `einsum-attention-scores`, `einops-rearrange-axis-split`

Implement `ex7_multihead_scores(q_packed, k_packed, h)`.

Inputs:
- `q_packed`: `(B, S, H*D)` — queries with the H heads packed into the last dim.
- `k_packed`: `(B, S, H*D)` — keys, same packing.
- `h`: int — number of heads (so per-head dim is `D = (H*D) / h`).

Steps:
1. Use `einops.rearrange` to split the packed dim: `(B, S, H*D) -> (B, H, S, D)` for both `q_packed` and `k_packed`. Print the shape of `q` after the split with prefix `q_split`.
2. Use `einops.einsum` with one pattern to compute scores: `scores[b, h, i, j] = sum_d q[b,h,i,d] * k[b,h,j,d]`. Print the shape of `scores` with prefix `scores`.
3. Return `scores` of shape `(B, H, S, S)`. Do **not** apply softmax or scale here.

The test verifies the prints, the shapes, and the values against an explicit per-head reference.

In [ ]:
def ex7_multihead_scores(q_packed: Tensor, k_packed: Tensor, h: int) -> Tensor:
    """Multi-head attention scores from packed (B, S, H*D) inputs.
    Must print `q_split shape=...` and `scores shape=...` debug lines."""
    raise NotImplementedError()


def _test_ex7():
    import io, contextlib

    B, S, H, D = 2, 4, 3, 5
    q_packed = t.randn(B, S, H * D)
    k_packed = t.randn(B, S, H * D)

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        scores = ex7_multihead_scores(q_packed, k_packed, h=H)
    log = buf.getvalue()
    print(log, end='')

    assert scores.shape == (B, H, S, S), f'expected ({B},{H},{S},{S}), got {scores.shape}'

    # Ground truth: split manually, then per-head matmul.
    q_ref = q_packed.reshape(B, S, H, D).permute(0, 2, 1, 3)  # (B, H, S, D)
    k_ref = k_packed.reshape(B, S, H, D).permute(0, 2, 1, 3)
    expected = q_ref @ k_ref.transpose(-2, -1)               # (B, H, S, S)
    assert t.allclose(scores, expected, atol=1e-5), 'scores differ from per-head reference'

    # Debug-print contract.
    assert 'q_split' in log, f'missing `q_split` print:\n{log}'
    assert 'scores' in log, f'missing `scores` print:\n{log}'
    assert f'({B}, {H}, {S}, {D})' in log or f'{(B, H, S, D)}' in log, (
        f'q_split shape not reported as ({B},{H},{S},{D}):\n{log}'
    )
    assert f'({B}, {H}, {S}, {S})' in log or f'{(B, H, S, S)}' in log, (
        f'scores shape not reported as ({B},{H},{S},{S}):\n{log}'
    )

    # Visualize per-head scores for batch 0: H subplots.
    fig, axes = plt.subplots(1, H, figsize=(3 * H, 3))
    if H == 1:
        axes = [axes]
    for hi in range(H):
        im = axes[hi].imshow(scores[0, hi].numpy(), cmap='coolwarm')
        axes[hi].set_title(f'head {hi}')
        axes[hi].set_xlabel('key j'); axes[hi].set_ylabel('query i')
        plt.colorbar(im, ax=axes[hi], fraction=0.046)
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex7')
    print("ex7 ✓")

_test_ex7()

<details><summary>Solution</summary>

```python
def ex7_multihead_scores(q_packed: Tensor, k_packed: Tensor, h: int) -> Tensor:
    q = rearrange(q_packed, 'b s (h d) -> b h s d', h=h)
    k = rearrange(k_packed, 'b s (h d) -> b h s d', h=h)
    print(f"q_split shape={tuple(q.shape)}")
    scores = einsum(q, k, 'b h i d, b h j d -> b h i j')
    print(f"scores shape={tuple(scores.shape)}")
    return scores
```

**Why rearrange + einsum, not one einsum.** einsum patterns don't split/merge axes — they only label and contract. So you need `rearrange` to do the head-split (`(h d) -> h d`) up front, then the einsum is a clean batched matmul over `d` with `b` and `h` both carried through.

**Reading the per-head plots.** Each head sees the same input but applies a different score pattern (in real models, that's because Q and K were projected through different weight matrices first). Here we skipped the projection — the heads only differ because their packed-dim slices are different — so the score heatmaps will look unrelated to each other.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex7'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex7',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()